In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import sys 
import os

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)


import tarfile
import urllib

# Get the data

In [ ]:
application_train_df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')
application_test_df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_test.csv')
application_train_df.head()

In [ ]:
bureau_df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/bureau.csv')
bureau_balance_df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv')
POS_CASH_balance_df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv')
credit_card_balance_df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv')
previous_application_df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/previous_application.csv')
installments_payments_df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv')


In [ ]:
# lets make some visualizations
import seaborn as sns
import math 

# distribution for the target variable (data imbalance)
fig, axs = plt.subplots(figsize=(7,5))
axs.pie(application_train_df["TARGET"].value_counts(), 
        labels = ["(0): No payment problems", "(1): Payment problems"],
       autopct = "%1.1f%%")
axs.set_title("Target Distribution")
plt.show()
plt.close()

print("--------------------------------------------------------------------------------")
print("Finantial information per target (raw)")
# lets see the total income distribution for each target 
fig, axs = plt.subplots(2,2,figsize=(13,11))
axs = axs.flatten()
# Log income
#sns.kdeplot(data=application_train_df, 
#    x=np.log1p(application_train_df["AMT_INCOME_TOTAL"]),
#    hue="TARGET",
#    common_norm=False,
#    ax=axs[0]
#)
#axs[0].set_xlabel("log(1 + Income)")

sns.kdeplot(data=application_train_df,
           x = "AMT_GOODS_PRICE",
           hue="TARGET",
           common_norm=False,
           ax=axs[0]
)
axs[0].set_xlabel("Goods Price")

# Income
sns.kdeplot(
    data=application_train_df[
        application_train_df["AMT_INCOME_TOTAL"] < 1_000_000
    ],
    x="AMT_INCOME_TOTAL",
    hue="TARGET",
    common_norm=False,
    ax=axs[1]
)
axs[1].set_xlabel("Income")

# Credit
sns.kdeplot(
    data=application_train_df,
    x="AMT_CREDIT",
    hue="TARGET",
    common_norm=False,
    ax=axs[2]
)
axs[2].set_xlabel("Credit Amount")

# Annuity
sns.kdeplot(
    data=application_train_df,
    x="AMT_ANNUITY",
    hue="TARGET",
    common_norm=False,
    ax=axs[3]
)
axs[3].set_xlabel("Annuity Amount")

plt.tight_layout()
plt.show()
plt.close()

print("--------------------------------------------------------------------------------")
print("Personal information per target (raw)")

fig, axs = plt.subplots(2,2,figsize=(13,11))
axs = axs.flatten()

# age distributions 
# lets to convert in years
application_train_df["AGE_YEARS"] = np.abs(application_train_df["DAYS_BIRTH"])/365
sns.kdeplot(
        data=application_train_df,
        x="AGE_YEARS",
        hue="TARGET",
        common_norm=False,
        ax=axs[0]
)
axs[0].set_xlabel("Age (Years)")

# employment information
application_train_df["EMPLOYED_YEARS"] = np.abs(application_train_df["DAYS_EMPLOYED"])/365
sns.kdeplot(
        data=application_train_df,
        x="EMPLOYED_YEARS",
        hue="TARGET",
        common_norm=False,
        ax=axs[1]
)
axs[1].set_xlabel("Employed (Years)")

sns.kdeplot(
        data=application_train_df,
        x="CNT_CHILDREN",
        hue="TARGET",
        common_norm=False,
        ax=axs[2]
)
axs[2].set_xlabel("Number of Children")

sns.kdeplot(
        data=application_train_df,
        x="CNT_FAM_MEMBERS",
        hue="TARGET",
        common_norm=False,
        ax=axs[3]
)
axs[3].set_xlabel("Family Members")

plt.show()
plt.close()

print("--------------------------------------------------------------------------------")
print("External risk scores")

fig, axs = plt.subplots(1,3,figsize=(15,6))
axs = axs.flatten()

sns.kdeplot(
    data=application_train_df,
    x = "EXT_SOURCE_1",
    hue = "TARGET",
    common_norm = False,
    ax=axs[0]
)
axs[0].set_xlabel("External Score 1")

sns.kdeplot(
    data=application_train_df,
    x = "EXT_SOURCE_2",
    hue = "TARGET",
    common_norm = False,
    ax=axs[1]
)
axs[1].set_xlabel("External Score 2")

sns.kdeplot(
    data=application_train_df,
    x = "EXT_SOURCE_3",
    hue = "TARGET",
    common_norm = False,
    ax=axs[2]
)

axs[2].set_xlabel("External Score 3")

plt.show()
plt.close()

print("--------------------------------------------------------------------------------")
print("Credit Bareau")

bareau_variables = ['AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY', 
               'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON',
               'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']

fig, axs = plt.subplots(3,2,figsize=(15,15))
axes = axs.flatten()

for i, (var, a) in enumerate(zip(bareau_variables, axes)):
    sns.kdeplot(
        data = application_train_df, 
        x = var,
        hue = "TARGET",
        common_norm = False,
        ax=a
    )
    a.set_xlabel(f"Index {var}")

plt.show()
plt.close()

In [ ]:
# lets plot some categorical variables

plt.figure(figsize=(4,4))
#contract type
sns.countplot(
    data=application_train_df,
    x="NAME_CONTRACT_TYPE",
    hue = 'TARGET'
)
plt.xticks(rotation=45)
plt.xlabel("Contract Type")
plt.show()
plt.close()

fig, axs = plt.subplots(3,1, figsize=(15,25))
axs = axs.flatten()
income_order = (
    application_train_df["NAME_INCOME_TYPE"].value_counts().index
)
#income type
sns.countplot(
    data=application_train_df,
    y="NAME_INCOME_TYPE",
    order=income_order,
    hue = 'TARGET',
    ax=axs[0]
)
axs[0].set_ylabel("Income Type")
# occupation type
occupation_order = (application_train_df["OCCUPATION_TYPE"].value_counts().index)
sns.countplot(
    data=application_train_df,
    y="OCCUPATION_TYPE",
    order=occupation_order,
    hue = 'TARGET',
    ax=axs[1]
)
axs[1].set_ylabel("Occupation Type")
# organization type
org_order = (
    application_train_df["ORGANIZATION_TYPE"].value_counts().head(20).index
)

sns.countplot(
    data=application_train_df[application_train_df["ORGANIZATION_TYPE"].isin(org_order)],
    y="ORGANIZATION_TYPE",
    order=org_order,
    hue = 'TARGET',
    ax=axs[2]
)

axs[2].set_ylabel("Organization Type")
plt.tight_layout()
plt.show()
plt.close()


In [ ]:
# lets handle with outliers

application_train_df["CNT_CHILDREN"] = (application_train_df["CNT_CHILDREN"].replace(19, np.nan))
application_train_df["DAYS_EMPLOYED"] = (application_train_df["DAYS_EMPLOYED"].replace(365243, np.nan))
application_train_df.loc[application_train_df["EMPLOYED_YEARS"] > 100, "EMPLOYED_YEARS"] = np.nan


In [ ]:
# lets overview the correlation with the target variable
corr_matrix = (application_train_df.select_dtypes(include="number").
               corr()["TARGET"].sort_values(key = np.abs, ascending = False))

In [ ]:
top_corr = corr_matrix.drop("TARGET").head(20)

plt.figure(figsize=(8,6))

sns.barplot(
    x=top_corr.values,
    y=top_corr.index
)

plt.xlabel("Correlation with TARGET")
plt.show()

## Feature Engine

In [ ]:
# finance variables 

application_train_df["DEBT_BURDEN"] = application_train_df["AMT_CREDIT"]/application_train_df["AMT_INCOME_TOTAL"]
application_train_df["PAYMENT_BURDEN"] = application_train_df["AMT_ANNUITY"]/application_train_df["AMT_INCOME_TOTAL"]

# lets see the total income distribution for each target 
fig, axs = plt.subplots(1,2,figsize=(13,7))
axs = axs.flatten()

# Credit
sns.kdeplot(
    data=application_train_df,
    x="DEBT_BURDEN",
    common_norm=False,
    ax=axs[0]
)
axs[0].set_xlabel("Credit Amount Burden")
axs[0].axvline(x = application_train_df["DEBT_BURDEN"].median(), color = 'red', linestyle = '--',
               label = f"Mean: {application_train_df["DEBT_BURDEN"].median():.2f}" )
axs[0].legend()

# Annuity
sns.kdeplot(
    data=application_train_df,
    x="PAYMENT_BURDEN",
    common_norm=False,
    ax=axs[1]
)
axs[1].set_xlabel("Annuity Amount burden")
axs[1].axvline(x = application_train_df["PAYMENT_BURDEN"].median(), color = 'red', linestyle = '--',
               label = f"Mean: {application_train_df["PAYMENT_BURDEN"].median():.2f}" )
axs[1].legend()

plt.tight_layout()
plt.show()
plt.close()

print(
    f"Customers typically request credit amounts equivalent to "
    f"{application_train_df['DEBT_BURDEN'].median():.2f} "
    f"times their annual income."
)

print(
   f"Customers typically must assume periodic payments equivalent to "
   f"{application_train_df['PAYMENT_BURDEN'].median():.2%} "
   f"of their annual income."
)



In [ ]:
# lets combine highly correlation variables

application_train_df["MEAN_EXT_SOURCES"] = (application_train_df[
                                        ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1))


### JOIN Information

In [ ]:
train_df = application_train_df.copy()

### External Credit History

In [ ]:
###################### bureau ####################################################
# this tell us the active credit 
active_bureau = bureau_df[bureau_df["CREDIT_ACTIVE"] == "Active"]
# this tell us the debt to credit ratio per customer for active 
debt_to_credit_ratio_bureau  = (
    active_bureau.groupby("SK_ID_CURR")["AMT_CREDIT_SUM_DEBT"].sum() /
    active_bureau.groupby("SK_ID_CURR")["AMT_CREDIT_SUM"].sum()
)
# this tell us the maximum days the customer delay
max_overdue_bureau = bureau_df.groupby("SK_ID_CURR")["CREDIT_DAY_OVERDUE"].max()

###################### bureau_balance ################################################
# Modify the status column 
bureau_balance_df["STATUS_MOD"] = (
    pd.to_numeric(bureau_balance_df["STATUS"], errors = "coerce")
    .fillna(0)
    .astype(int)
)
# get for each SK_ID_CURR the worst status
worst_status_curr = (
    bureau_balance_df.groupby("SK_ID_BUREAU")["STATUS_MOD"]
    .max()
    .reset_index()
    .merge(bureau_df[["SK_ID_BUREAU", "SK_ID_CURR"]], on="SK_ID_BUREAU")
    .groupby("SK_ID_CURR")["STATUS_MOD"]
    .max()
    .rename("WORST_STATUS")   
)

########################## Merge bureau info into train_data #############################
debt_to_credit = debt_to_credit_ratio_bureau.rename("DEBT_TO_CREDIT_RATIO").reset_index()
max_overdue    = max_overdue_bureau.rename("MAX_OVERDUE_DAYS").reset_index()
worst_status   = worst_status_curr.rename("WORST_STATUS").reset_index()
# merge everything
bureau_features = (
    debt_to_credit
    .merge(max_overdue,  on="SK_ID_CURR", how="outer")
    .merge(worst_status, on="SK_ID_CURR", how="outer")
)
# merge into train data
train_df = train_df.merge(bureau_features, on="SK_ID_CURR", how="left")

In [ ]:
############# previous application ####################

## get the previous approval rate
previous_info = previous_application_df.groupby("SK_ID_CURR").agg(
    PREV_COUNT   = ("SK_ID_PREV", "count"),
    PREV_APPROVED_COUNT   = ("NAME_CONTRACT_STATUS", lambda x: (x == "Approved").sum()),
)

previous_info["PREV_APPROVAL_RATE"] = (
    prev_info["PREV_APPROVED_COUNT"] / prev_info["PREV_COUNT"]
)